In [0]:
# scope: sqlserver-creds, username as key and password, sensitive
username = dbutils.secrets.get(scope="sqlserver-creds", key="username")
print ("username", username) # READACTED 

password = dbutils.secrets.get(scope="sqlserver-creds", key="password")
print ("password", password)  # READACTED
 

In [0]:
# but still password is visible, you should avoid accessing by char array

print (password[0], password[1])

# sensitive ?? DONT DO, DONT PRINT 
for c in password: print (c)
print ('----')

In [0]:
DATABASE = "free-sql-db-5202703"
HOSTNAME = "gks-test.database.windows.net"

# f-string

jdbc_url = f"jdbc:sqlserver://{HOSTNAME}:1433;database={DATABASE};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

jdbc_properties = {
    "user": username ,
    "password": password ,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


In [0]:
df_products = spark.read.jdbc(
    url=jdbc_url,
    table="SalesLT.product",
    properties=jdbc_properties
)

df_products.display()

In [0]:
storage_account = "gopaldatalake"
container = "gold"
folder = "popular-movies"

abfss_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{folder}/"

popularMoviesDf = spark.read.parquet(abfss_path)
popularMoviesDf.printSchema()
popularMoviesDf.show(2)


In [0]:
jdbc_url = f"jdbc:sqlserver://{HOSTNAME}:1433;database={DATABASE};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

popularMoviesDf.write \
  .format("jdbc") \
  .option("url", jdbc_url) \
  .option("dbtable", "gks_popular_movies") \
  .option("user", username) \
  .option("password", password) \
  .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
  .mode("overwrite") \
  .save()

In [0]:
jdbc_url = f"jdbc:sqlserver://{HOSTNAME}:1433;database={DATABASE};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

df = spark.read \
  .format("jdbc") \
  .option("url", jdbc_url) \
  .option("dbtable", "gks_popular_movies") \
  .option("user", username) \
  .option("password", password) \
  .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
  .load()

df.printSchema()
df.show(2)